<a href="https://colab.research.google.com/github/vish9571/web-scraping-sql-pipeline/blob/v1-scraping/etl_web_scraper_to_sql.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Step 1: Install the necessary libraries (Run this if you don't have them)
# !pip install requests beautifulsoup4 pandas

# Step 2: Import the libraries
import requests
from bs4 import BeautifulSoup

# Step 3: Define the website URL we want to scrape
url = "https://books.toscrape.com/"

# Step 4: Send a "GET" request to the website (like typing it into your browser)
response = requests.get(url)

# Step 5: Check if the request was successful (Code 200 means OK!)
if response.status_code == 200:
    print("Success! We connected to the website.\n")

    # Step 6: Parse the HTML content using BeautifulSoup
    soup = BeautifulSoup(response.text, 'html.parser')

    # Let's print out the title of the webpage just to prove we grabbed it!
    page_title = soup.title.text
    print(f"The title of the webpage is: {page_title}")

else:
    print(f"Failed to connect. Status code: {response.status_code}")

Success! We connected to the website.

The title of the webpage is: 
    All products | Books to Scrape - Sandbox



In [ ]:
import requests
from bs4 import BeautifulSoup

url = "https://books.toscrape.com/"
response = requests.get(url)
soup = BeautifulSoup(response.text, 'html.parser')

# 1. Find ALL the book containers on the page
books = soup.find_all('article', class_='product_pod')

print(f"Success! Found {len(books)} books on the homepage.\n")
print("Extracting data for the first 5 books...\n")

# 2. Start a loop to go through just the first 5 books (to keep our screen clean)
for book in books[:5]:

    # Extract the Title: It's hidden inside an <h3> tag, inside an <a> (link) tag
    title = book.h3.a['title']

    # Extract the Price: It's inside a <p> tag with a specific class
    price = book.find('p', class_='price_color').text

    # Extract the Rating: This is a bit tricky! The rating is stored as a class name (e.g., "star-rating Three")
    rating_element = book.find('p', class_='star-rating')
    rating = rating_element['class'][1] # This grabs the second word (e.g., "Three")

    # 3. Print out our clean data
    print(f"Book Title: {title}")
    print(f"Price:      {price}")
    print(f"Rating:     {rating} stars")
    print("-" * 40)

Success! Found 20 books on the homepage.

Extracting data for the first 5 books...

Book Title: A Light in the Attic
Price:      Â£51.77
Rating:     Three stars
----------------------------------------
Book Title: Tipping the Velvet
Price:      Â£53.74
Rating:     One stars
----------------------------------------
Book Title: Soumission
Price:      Â£50.10
Rating:     One stars
----------------------------------------
Book Title: Sharp Objects
Price:      Â£47.82
Rating:     Four stars
----------------------------------------
Book Title: Sapiens: A Brief History of Humankind
Price:      Â£54.23
Rating:     Five stars
----------------------------------------


In [ ]:
import pandas as pd

# 1. We'll store our results in a list of dictionaries
book_data = []

for book in books:
    title = book.h3.a['title']

    # Grab the raw price string
    raw_price = book.find('p', class_='price_color').text

    # CLEANING: Keep only the numbers and the dot (remove Â£)
    clean_price = raw_price.replace('Â£', '').replace('£', '')

    rating_word = book.find('p', class_='star-rating')['class'][1]

    # CLEANING: Map words to numbers
    rating_map = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
    numeric_rating = rating_map.get(rating_word, 0)

    # Append to our list
    book_data.append({
        "Title": title,
        "Price_GBP": float(clean_price),
        "Star_Rating": numeric_rating
    })

# 2. Convert the list to a Pandas DataFrame
df = pd.DataFrame(book_data)

# 3. Show the first 5 rows and the "types" of our data
print("--- Cleaned Data Table ---")
print(df.head())
print("\n--- Column Data Types ---")
print(df.dtypes)

--- Cleaned Data Table ---
                                   Title  Price_GBP  Star_Rating
0                   A Light in the Attic      51.77            3
1                     Tipping the Velvet      53.74            1
2                             Soumission      50.10            1
3                          Sharp Objects      47.82            4
4  Sapiens: A Brief History of Humankind      54.23            5

--- Column Data Types ---
Title           object
Price_GBP      float64
Star_Rating      int64
dtype: object


In [ ]:
import sqlite3

# 1. Create a connection to a new database file
# If the file doesn't exist, Python will create it automatically
conn = sqlite3.connect('books_collection.db')

# 2. Use Pandas to send the data to a SQL table named 'books'
# 'if_exists=replace' ensures that if you run this twice, it just refreshes the table
df.to_sql('books', conn, if_exists='replace', index=False)

print("Success! The data has been injected into the SQL database.\n")

# 3. Let's verify it worked by running a simple SQL test query
# We will select only the books that cost more than 50 GBP
query = "SELECT * FROM books WHERE Price_GBP > 50"

# Read the SQL result back into a new DataFrame
expensive_books = pd.read_sql(query, conn)

print("--- SQL Query Result: Books Over £50 ---")
print(expensive_books.head())

# Always close the connection when you're done
conn.close()

Success! The data has been injected into the SQL database.

--- SQL Query Result: Books Over £50 ---
                                   Title  Price_GBP  Star_Rating
0                   A Light in the Attic      51.77            3
1                     Tipping the Velvet      53.74            1
2                             Soumission      50.10            1
3  Sapiens: A Brief History of Humankind      54.23            5
4                        The Black Maria      52.15            1


In [ ]:
import sqlite3
import pandas as pd

# Reconnect to our database
conn = sqlite3.connect('books_collection.db')

print("--- Day 5: Extracting Business Insights via SQL ---\n")

# Query 1: What is the average price for each rating?
# This shows if higher-rated books are actually more expensive.
query_1 = """
SELECT Star_Rating,
       COUNT(*) as Total_Books,
       ROUND(AVG(Price_GBP), 2) as Avg_Price
FROM books
GROUP BY Star_Rating
ORDER BY Star_Rating DESC
"""
print("1. Average Price by Star Rating:")
print(pd.read_sql(query_1, conn))
print("\n" + "="*40 + "\n")

# Query 2: High Quality at a Low Price (The "Value" List)
query_2 = """
SELECT Title, Price_GBP
FROM books
WHERE Star_Rating = 5 AND Price_GBP < 20
ORDER BY Price_GBP ASC
"""
print("2. Five-Star Books under £20:")
print(pd.read_sql(query_2, conn))
print("\n" + "="*40 + "\n")

# Query 3: Total Inventory Statistics
query_3 = """
SELECT COUNT(*) as Total_Inventory_Count,
       SUM(Price_GBP) as Total_Inventory_Value,
       MAX(Price_GBP) as Most_Expensive,
       MIN(Price_GBP) as Cheapest
FROM books
"""
print("3. Overall Inventory Summary:")
summary = pd.read_sql(query_3, conn)
print(summary)

conn.close()

--- Day 5: Extracting Business Insights via SQL ---

1. Average Price by Star Rating:
   Star_Rating  Total_Books  Avg_Price
0            5            4      39.75
1            4            4      31.11
2            3            3      42.32
3            2            3      36.83
4            1            6      40.02


2. Five-Star Books under £20:
         Title  Price_GBP
0  Set Me Free      17.46


3. Overall Inventory Summary:
   Total_Inventory_Count  Total_Inventory_Value  Most_Expensive  Cheapest
0                     20                 760.97           57.25     13.99
